# 노후주택 안전등급 분류 학습 (우수/보통/불량, Colab)

탐지(YOLO)로는 등급 판정이 안 돼서(휴리스틱 22%), **등급을 이미지 분류로 직접 학습**한다.

- 데이터: `house_grade_cls.zip` (train/val 아래 good·fair·poor 폴더) — Mac에서 생성
- 클래스 3종: good(우수) / fair(보통) / poor(불량)
- 탐지 모델은 '균열 위치 박스' 시각화용으로 별도 유지

**런타임 → GPU(A100/L4)** 설정 후 위에서부터 실행.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 드라이브 연결 + 데이터 압축해제
`house_grade_cls.zip` 을 My Drive 최상위에 올린 뒤 실행. train/val 개수가 뜨면 성공.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob
ZIP_PATH = '/content/drive/MyDrive/house_grade_cls.zip'
assert os.path.exists(ZIP_PATH), f'파일 없음: {ZIP_PATH}'
!rm -rf /content/house_grade_cls
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content')

roots = glob.glob('/content/**/house_grade_cls', recursive=True) + ['/content/house_grade_cls']
DATA = next(r for r in roots if os.path.isdir(os.path.join(r, 'train')))
print('데이터 루트:', DATA)
for split in ('train', 'val'):
    print(split, {os.path.basename(d): len(os.listdir(d)) for d in glob.glob(f'{DATA}/{split}/*')})

## 3. 등급 분류 학습
YOLO11m-cls. 불량(poor)이 많은 불균형이라 val의 **클래스별 정확도**를 꼭 확인.

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO('yolo11m-cls.pt')
results = model.train(
    data=DATA,
    epochs=50,
    imgsz=320,     # 결함 디테일 위해 분류 기본(224)보다 크게
    batch=64,
    patience=15,
    name='house_grade',
)

## 4. best.pt 저장 (다운로드 + 드라이브 백업)

In [ ]:
import os, shutil
run_dir = results.save_dir
best = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', best)
shutil.copy(best, '/content/drive/MyDrive/house_grade_best.pt')
print('드라이브 저장: /content/drive/MyDrive/house_grade_best.pt')
from google.colab import files
files.download(best)

## 5. ★ 등급 실증 — confusion matrix + 위험 누락률
탐지+휴리스틱(22%) 대비 얼마나 좋아졌는지. 특히 **불량 재현율**(실제 불량을 불량으로)이 핵심 안전 지표.

In [ ]:
import glob, os
from collections import defaultdict

KO = {'good': '우수', 'fair': '보통', 'poor': '불량'}
GROUPS = ['우수', '보통', '불량']
best_model = YOLO(best)
cm = defaultdict(int); n = 0
for cls_dir in glob.glob(f'{DATA}/val/*'):
    true_g = KO[os.path.basename(cls_dir)]
    for img in glob.glob(f'{cls_dir}/*'):
        r = best_model.predict(img, imgsz=320, verbose=False)[0]
        pred_g = KO[best_model.names[int(r.probs.top1)]]
        cm[(true_g, pred_g)] += 1; n += 1

print(f'검증 {n}장\n실제\\예측 |', ' | '.join(f'{g:>4}' for g in GROUPS))
for t in GROUPS:
    print(f'{t:>6}   |', ' | '.join(f'{cm[(t,p)]:>4}' for p in GROUPS))
acc = sum(cm[(g,g)] for g in GROUPS)/n
dtot = sum(cm[('불량',p)] for p in GROUPS)
missed = cm[('불량','우수')] + cm[('불량','보통')]
print(f'\n등급 정확도: {acc:.1%}  (탐지+휴리스틱 22.3% 대비)')
print(f'불량 재현율: {cm[("불량","불량")]/dtot:.1%} | 위험누락률: {missed/dtot:.1%}')